In [2]:
from ast import Module
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn


In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/master/data.csv')

In [4]:
df.drop(columns=['id' , 'Unnamed: 32'] , inplace=True)

In [5]:
X_train , x_test , y_train , y_test = train_test_split(df.iloc[:,1:] , df.iloc[:,0] , test_size=0.2)

In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
x_test = scaler.transform(x_test)

In [7]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [8]:
X_train_tensor = torch.from_numpy(X_train)
x_test_tensor = torch.from_numpy(x_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [9]:
from torch.utils.data import Dataset,DataLoader

In [10]:
class customDataSet(Dataset):
  def __init__(self , features , labels):

    self.features = features
    self.labels = labels

  def __len__(self):

    return self.features.shape[0]

  def __getitem__(self, index):

    return self.features[index] , self.labels[index]


In [11]:
train_dataset = customDataSet(X_train_tensor , y_train_tensor)
test_dataset = customDataSet(x_test_tensor , y_test_tensor)

## now dataloaders

In [14]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True)

## now define the model

In [15]:
learning_rate = 0.1
epochs = 25

In [16]:
class neuralNetwork(nn.Module):

  def __init__(self , n_features):

    super().__init__()

    self.linear = nn.Linear(n_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, X):

    out = self.linear(X)
    out = self.sigmoid(out)

    return out

In [18]:
model = neuralNetwork(X_train_tensor.shape[1])


In [21]:
loss_function = nn.BCELoss()

optimizer = torch.optim.SGD(model.parameters() , lr=learning_rate)

In [28]:
# loop
for epoch in range(epochs):


  for batch_features , batch_labels in train_dataloader:

# forward pass w.x + b
    y_pred = model(batch_features.float())


# calc loss
    loss = loss_function(y_pred.float(), batch_labels.view(-1,1).float())


# clear the gradients
    optimizer.zero_grad()

# backward pass
    loss.backward()

# parameters update


    # with torch.no_grad():
    optimizer.step()
    #   model.weights -= learning_rate * model.weights.grad
    #   model.bias -= learning_rate * model.bias.grad

    # # zero grads

    #   model.weights.grad.zero_()
    #   model.bias.grad.zero_()

    print(f"Epoch : {epoch + 1} and loss : {loss.item()} ")






Epoch : 1 and loss : 0.7792990207672119 
Epoch : 1 and loss : 0.5751098990440369 
Epoch : 1 and loss : 0.5090848207473755 
Epoch : 1 and loss : 0.39552026987075806 
Epoch : 1 and loss : 0.29564160108566284 
Epoch : 1 and loss : 0.33755218982696533 
Epoch : 1 and loss : 0.347037672996521 
Epoch : 1 and loss : 0.2541998028755188 
Epoch : 1 and loss : 0.24443109333515167 
Epoch : 1 and loss : 0.22527915239334106 
Epoch : 1 and loss : 0.2124955803155899 
Epoch : 1 and loss : 0.24391056597232819 
Epoch : 1 and loss : 0.2365933209657669 
Epoch : 1 and loss : 0.15292048454284668 
Epoch : 1 and loss : 0.14637520909309387 
Epoch : 2 and loss : 0.16387832164764404 
Epoch : 2 and loss : 0.14448411762714386 
Epoch : 2 and loss : 0.14760209619998932 
Epoch : 2 and loss : 0.18696846067905426 
Epoch : 2 and loss : 0.23384200036525726 
Epoch : 2 and loss : 0.1248011440038681 
Epoch : 2 and loss : 0.13951395452022552 
Epoch : 2 and loss : 0.22072052955627441 
Epoch : 2 and loss : 0.13479605317115784 
E

## evaluation

In [34]:
model.eval()
accuracy_list = []

In [37]:
with torch.no_grad():
  for batch_features , batch_labels in test_dataloader:

   # forward propagation
   y_pred = model(batch_features.float())

   y_pred = (y_pred > 0.9).float() # convert to binary

   batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
   accuracy_list.append(batch_accuracy)

   overall_accuracy = sum(accuracy_list) / len(accuracy_list)
   print(f"Accuracy : {overall_accuracy}")

Accuracy : 0.9479166666666666
Accuracy : 0.9296875
Accuracy : 0.93125
Accuracy : 0.9427083333333334


In [ ]:
from sklearn.datasets import make_classification
import torch

In [ ]:
X , y = make_classification(
    n_samples=10,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_classes=2,
    random_state=42
)

In [ ]:
X


In [ ]:
y

In [ ]:
X = torch.tensor(X , dtype=torch.float64)
y = torch.tensor(y , dtype=torch.long)

In [ ]:
from torch.utils.data import Dataset,DataLoader

In [ ]:

class customDataSet(Dataset):
  def __init__(self , features , labels):

    self.features = features
    self.labels = labels

  def __len__(self):

    return self.features.shape[0]

  def __getitem__(self, index):

    return self.features[index] , self.labels[index]



In [ ]:
custom = customDataSet(X , y)

In [ ]:
custom[3]

### A note about samplers
Sampler
In PyTorch, the sampler in the DataLoader determines the strategy for selecting samples from the dataset during data loading. It controls how indices of the dataset are drawn for each batch.
0123456789
Types of Samplers
PyTorch provides several predefined samplers, and you can create custom ones:
46 13 7 8 2 0 9 5
1. SequentialSampler:
α
Samples elements sequentially, in the order they appear in the dataset.
Default when shuffle=False.
2. RandomSampler:
Samples elements randomly without replacement.
Default when shuffle=True.

### DataLoader Important Parameters
The DataLoader class in PyTorch comes with several parameters that allow you to customize how data is loaded, batched, and preprocessed. Some of the most commonly used and important parameters include:
1. dataset (mandatory):
The Dataset from which the DataLoader will pull data.
Must be a subclass of torch.utils.data.Dataset that implements len. plements_getitem_and
2. batch size:
How many samples per batch to load.
Default is 1.
Larger batch sizes can speed up training on GPUs but require more memory.
shuffle:
If True, the DataLoader will shuffle the dataset indices each epoch.
Helpful to avoid the model becoming too dependent on the order of samples.
4. num workers:
The number of worker processes used to load data in parallel.
Setting num_workers > 0 can speed up data loading by leveraging multiple CPU cores, especially if I/O or preprocessing is a bottleneck.
5. pin_memory:
If True, the DataLoader will copy tensors into pinned (page-locked) memory before returning them.
This can improve GPU transfer speed and thus overall training throughput, particularly on CUDA systems.
6. drop_last:
If True, the DataLoader will drop the last incomplete batch if the total number of
systems.
6. drop last
If True, the DataLoader will drop the last incomplete batch if the total number of samples is not divisible by the batch size.
Useful when exact batch sizes are required (for example, in some batch normalization scenarios).
7. collate_fn:
A callable that processes a list of samples into a batch (the default simply stacks tensors).
Custom collate_fn can handle variable-length sequences, perform custom batching logic, or handle complex data structures.
8. sampler:
sampler defines the strategy for drawing samples (e.g., for handling imbalanced classes, or custom sampling strategies).
batch_sampler works at the batch level, controlling how batches are formed.
Typically, you don't need to specify these if you are using batch_size and shuffle. However, they provide lower-level control if you have advanced requirements.